# VECTOR RUN: canonical game × Decision 2.0 Colab benchmark

このNotebookは7モデルのうち1つを1つの実GPU runtimeで測定します。Kai / Eos / Sol / Sol-Reasoning / Nox / Lux / Vegaそれぞれに専用Colab sessionと一意のCLI configを用意し、7 sessionを並列に動かしてください。WSL側の認証は `/home/makim/.local/bin/colab --auth adc` を使い、Notebookにはcredentialを書きません。

ゲームは外部の [vector-run-benchmark](https://github.com/Sunwood-ai-labs/vector-run-benchmark) から固定SHAでcheckoutします。このrepositoryにゲームsourceをコピーしません。ready status、正式な `measurementCommit`、確定wire contract、trace schema validatorのpassがそろうまで、Notebookはbootstrap/GPU準備へ進みません。

`--questions` は1回のSystem One API callに含める同時質問数です。q3は `action` / `commit` / `danger`、q64はその3問と `p0`〜`p60` の61 `noul` probesです。dispatch回数ではありません。ゲームは120Hz、16tickごとに判断し、wait/jump/releaseを使います。各系列は30秒・最大2 jumpのcensored条件で、clearやcollision完了を意味しません。3×3 movieはゲームrepositoryが所有するrenderer経由で作成します。

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import re
import subprocess
import sys
import zipfile

BENCHMARK_REPOSITORY = "https://github.com/Sunwood-ai-labs/vector-run-decision-colab.git"
GAME_REPOSITORY = "https://github.com/Sunwood-ai-labs/vector-run-benchmark.git"
BENCHMARK_COMMIT = "b2e8fcbbf4b4f68931e4f9bbfbd434023348d3f0"  # 親が共有するrunner統合commitへ更新
GAME_COMMIT = "717f02dc9852b88c253ace32f42fcb6d780ed0d3"       # handoff measurementCommitと一致するSHAのみ
MODEL = "kai"           # kai, eos, sol, sol-reasoning, nox, lux, vega
MODEL_CHOICES = ("kai", "eos", "sol", "sol-reasoning", "nox", "lux", "vega")
# 各担当がruntime/session evidenceから入力する。Noneはunknownで、model名だけから推測しない。
RUNTIME_HISTORY = {
    "model": MODEL,
    "runtimeReused": None,
    "weightsCached": None,
    "priorModelLoaded": None,
    "priorSystemOneCall": None,
    "priorInference": None,
    "priorPrototypeGameRun": None,
    "evidence": {
        "runtimeReused": "unknown; add session/runtime evidence",
        "weightsCached": "unknown; add cache evidence",
        "priorModelLoaded": "unknown; add runtime evidence",
        "priorSystemOneCall": "unknown; add request evidence",
        "priorInference": "unknown; add inference evidence",
        "priorPrototypeGameRun": "unknown; add game-run evidence",
    },
}
MAX_SECONDS = 30
MAX_JUMPS = 2
SHA_PATTERN = re.compile(r"^[0-9a-fA-F]{40}$")

def require_sha(name, value):
    if not isinstance(value, str) or not SHA_PATTERN.fullmatch(value):
        raise RuntimeError(f"{name} must be a full 40-character commit SHA; canonical game measurementCommit is not ready")

if MODEL not in MODEL_CHOICES:
    raise RuntimeError(f"MODEL must be one of {MODEL_CHOICES}")
require_sha("GAME_COMMIT", GAME_COMMIT)
require_sha("BENCHMARK_COMMIT", BENCHMARK_COMMIT)

In [ ]:
def git(path, *args):
    return subprocess.check_output(["git", *args], cwd=path, text=True, stderr=subprocess.STDOUT).strip()

def canonical_repo(url):
    return url.removesuffix(".git").rstrip("/").lower()

def checkout_pinned(repository, destination, commit):
    if destination.exists():
        if not (destination / ".git").exists():
            raise RuntimeError(f"Refusing to replace a non-Git directory: {destination}")
        origin = git(destination, "remote", "get-url", "origin")
        if canonical_repo(origin) != canonical_repo(repository):
            raise RuntimeError(f"Unexpected origin in {destination}")
        if git(destination, "status", "--porcelain"):
            raise RuntimeError(f"Refusing to checkout over a dirty worktree: {destination}")
    else:
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", repository, str(destination)], check=True)
    subprocess.run(["git", "fetch", "origin", commit], cwd=destination, check=True)
    subprocess.run(["git", "checkout", "--detach", commit], cwd=destination, check=True)
    actual = git(destination, "rev-parse", "HEAD")
    if actual.lower() != commit.lower():
        raise RuntimeError(f"Immutable checkout mismatch: requested {commit}, got {actual}")
    if git(destination, "status", "--porcelain"):
        raise RuntimeError(f"Pinned checkout is not clean: {destination}")
    return destination

root = checkout_pinned(BENCHMARK_REPOSITORY, Path("/content/vector-run-decision-colab"), BENCHMARK_COMMIT)
sys.path.insert(0, str(root))
from experiments.colab.run_bench import validate_handoff, validate_runtime_history
handoff_path = root / "handoffs" / "canonical-game.json"
handoff = json.loads(handoff_path.read_text(encoding="utf-8"))
handoff_errors = validate_handoff(handoff, GAME_COMMIT)
if handoff_errors:
    raise RuntimeError("Canonical handoff is not measurement-ready: " + "; ".join(handoff_errors))
runtime_history, history_errors = validate_runtime_history(RUNTIME_HISTORY, MODEL)
if history_errors:
    raise RuntimeError("Runtime history input is invalid: " + "; ".join(history_errors))
game_root = checkout_pinned(GAME_REPOSITORY, Path("/content/vector-run-game"), GAME_COMMIT)
if not (game_root / "scripts" / "decision-bench.mjs").is_file():
    raise RuntimeError("Pinned game checkout does not contain scripts/decision-bench.mjs")
print({"benchmarkCommit": git(root, "rev-parse", "HEAD"), "gameCommit": git(game_root, "rev-parse", "HEAD"), "handoffGate": "passed"})

In [ ]:
environment = os.environ.copy()
environment["VECTOR_RUN_ROOT"] = str(root)
subprocess.run(["bash", "experiments/colab/bootstrap.sh"], cwd=root, env=environment, check=True)
uv = str(Path.home() / ".local" / "bin" / "uv")
if not Path(uv).is_file():
    uv = "uv"
subprocess.run([
    uv, "run", "--no-project", "--no-managed-python", "--no-sync", "python", "-c",
    "import torch; assert torch.cuda.is_available(), 'CUDA GPU unavailable'; print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0)})",
], cwd=root, check=True)

In [ ]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_dir = Path("/content/vector-run-results") / MODEL / run_id
output_dir.mkdir(parents=True, exist_ok=False)
runtime_history_path = Path("/content") / f"runtime-history-{MODEL}-{run_id}.json"
runtime_history_path.write_text(json.dumps(runtime_history, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
command = [
    uv, "run", "--no-project", "--no-managed-python", "--no-sync", "python", "experiments/colab/run_bench.py",
    "--game-dir", str(game_root), "--game-commit", GAME_COMMIT,
    "--benchmark-commit", BENCHMARK_COMMIT,
    "--models", "single", "--model", MODEL,
    "--runtime-history", str(runtime_history_path),
    "--output-dir", str(output_dir),
]
completed = subprocess.run(command, cwd=root)
print({"runnerProcessExit": completed.returncode, "aggregateMeasurementStatus": "not_aggregated", "outputFiles": sorted(path.name for path in output_dir.glob("*.json"))})

In [ ]:
from google.colab import files
json_files = sorted(output_dir.glob("*.json"))
if not json_files:
    raise RuntimeError("No JSON report was produced; inspect the runner output before claiming a measurement")
archive = Path("/content") / f"vector-run-{MODEL}-{run_id}.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zipped:
    for path in json_files:
        zipped.write(path, arcname=path.name)
files.download(str(archive))

## 3×3 movie route

Download the raw `{model}-q3.game.json` reports from each model's dedicated session. Movie rendering belongs to the same pinned external game checkout, which owns the 3×3 player and capture CLI. The current handoff marks the trace schema pending; once the game owner publishes the renderer entry point, pass the downloaded reports to that game-owned CLI. This notebook does not bundle or call the old prototype player, and the movie must be labeled as trace playback of 30-second censored runs.